# Measuring a Classifier

**CS 212 AI Programming 1**

This is the Jupyter notebook version of the [Lab 4 example](https://lcc-cit.github.io/CS212-CourseMaterials/Examples/Lab04-ClassifierMetrics/CS212_Lab04_Example.html). It builds on the spam detection example (see the [Spam Detection overview](https://lcc-cit.github.io/CS212-CourseMaterials/Examples/SpamDetection/CS212_SpamDetection_Overview.html)). It takes the spam classifier you already know and shows what its performance metrics mean and how they are calculated.

*Accuracy*, *precision*, *recall* and *F1* are all **performance metrics**: numbers that measure how well a classifier does its job. By the end of this notebook you will have calculated each of them by hand and checked your answers against scikit-learn.

## 1. Starting Point: The Spam Classifier

The code cell below repeats the steps of the Spam Detection example. It loads the SMS Spam Collection (5,574 real text messages that people labeled as spam or ham), holds back 25% of the messages for testing, counts the words, trains a naive Bayes classifier, and classifies the test messages. We won't explain those steps again. The Spam Detection notebook explains each one.

There are two small differences. We read the data with Python's `csv` module, and we use numbers for the labels: **1 means spam and 0 means ham**.

In [1]:
import csv
import io
from urllib.request import urlopen

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

DATA_URL = "https://raw.githubusercontent.com/LCC-CIT/CS212-CourseMaterials/main/docs/Examples/SpamDetection/SpamDetectionNotebook/sms_spam.csv"

# Read the data. Each row has a label (spam or ham) and the text of one message.
messages = []
labels = []
with urlopen(DATA_URL) as response:
    text = response.read().decode("utf-8")
for row in csv.DictReader(io.StringIO(text)):
    messages.append(row["message"])
    if row["label"] == "spam":
        labels.append(1)
    else:
        labels.append(0)

# The same split, word counts and classifier as the Spam Detection example.
train_messages, test_messages, train_labels, test_labels = train_test_split(
    messages, labels, test_size=0.25, random_state=42)

vectorizer = CountVectorizer()
train_counts = vectorizer.fit_transform(train_messages)
test_counts = vectorizer.transform(test_messages)

classifier = MultinomialNB()
classifier.fit(train_counts, train_labels)
predicted_labels = classifier.predict(test_counts).tolist()   # a plain list of 0s and 1s

print(len(messages), "messages,", sum(labels), "of them are spam")
print(len(test_messages), "test messages,", sum(test_labels), "of them are spam")

5574 messages, 747 of them are spam
1394 test messages, 191 of them are spam


## 2. What We Are Measuring

To find out how well a classifier works, you compare its answers with the right answers for the same messages. Both are needed:

- The **true labels** are the correct classes. People decided them: someone read each message and marked it spam or ham.
- The **predicted labels** are the classes the classifier chose for the same messages.

We measure the classifier on the *test* messages, which it did not learn from. Here are the first five test messages that really are spam, with the label that the classifier predicted for each:

In [2]:
shown = 0
for i in range(len(test_messages)):
    if test_labels[i] == 1 and shown < 5:
        print("true:", test_labels[i], "| predicted:", predicted_labels[i], "|", test_messages[i][:60])
        shown += 1

true: 1 | predicted: 1 | FREE MESSAGE Activate your 500 FREE Text Messages by replyin
true: 1 | predicted: 1 | T-Mobile customer you may now claim your FREE CAMERA PHONE u
true: 1 | predicted: 1 | Kit Strip - you have been billed 150p. Netcollex Ltd. PO Box
true: 1 | predicted: 1 | URGENT This is our 2nd attempt to contact U. Your £900 prize
true: 1 | predicted: 1 | Call Germany for only 1 pence per minute! Call from a fixed 


## 3. Why Accuracy Isn't Enough

*Accuracy* is the percent of the messages that the classifier got right. `accuracy_score` in scikit-learn calculates it. We'll compare the spam classifier with a "classifier" that never needs any training: it calls every message ham.

In [3]:
from sklearn.metrics import accuracy_score

always_ham = [0] * len(test_labels)

print("Spam classifier:", round(accuracy_score(test_labels, predicted_labels), 3))
print("Always ham:     ", round(accuracy_score(test_labels, always_ham), 3))

Spam classifier: 0.986
Always ham:      0.863


The spam classifier's accuracy of 98.6% sounds excellent, but the "always ham" classifier is already 86.3% accurate, because most messages are ham. It also never finds a single spam message, which makes it useless as a spam filter. And if spam were much rarer than 13%, the problem would be worse: a classifier that is 99% accurate when 99% of the messages are ham is no better than "always ham".

To see how well a classifier does on the rare class, we need to count the different kinds of right and wrong answers.

## 4. The Four Counts

Pick one class to be the *positive* class. Here it is spam (label 1). Every prediction is one of four kinds:

| | Predicted spam | Predicted ham |
| --- | --- | --- |
| **Really spam** | True positive (TP) | False negative (FN) |
| **Really ham** | False positive (FP) | True negative (TN) |

A table like this is called a *confusion matrix*. This function counts the four kinds with a plain `for` loop:

In [4]:
def count_results(true_labels, predicted_labels):
    """Count the four kinds of result. The positive class is 1 (spam)."""
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    for i in range(len(true_labels)):
        if predicted_labels[i] == 1 and true_labels[i] == 1:
            true_positives += 1
        elif predicted_labels[i] == 1 and true_labels[i] == 0:
            false_positives += 1
        elif predicted_labels[i] == 0 and true_labels[i] == 1:
            false_negatives += 1
        else:
            true_negatives += 1
    return true_positives, false_positives, false_negatives, true_negatives


tp, fp, fn, tn = count_results(test_labels, predicted_labels)
print("True positives: ", tp, "(spam that it called spam)")
print("False positives:", fp, "(ham that it called spam)")
print("False negatives:", fn, "(spam that it called ham)")
print("True negatives: ", tn, "(ham that it called ham)")

True positives:  178 (spam that it called spam)
False positives: 6 (ham that it called spam)
False negatives: 13 (spam that it called ham)
True negatives:  1197 (ham that it called ham)


## 5. Calculating the Performance Metrics

From the four counts you can calculate these performance metrics:

| Measure | Formula | The question it answers |
| --- | --- | --- |
| Accuracy (percent correct) | (TP + TN) / all messages | What percent of all the messages did it get right? |
| Precision | TP / (TP + FP) | When it says "spam", how often is it right? |
| Recall | TP / (TP + FN) | Of the real spam messages, how many did it find? |
| F1 | 2 × precision × recall / (precision + recall) | One number that is high only when precision *and* recall are both high |

The functions below are careful about dividing by zero. For example, a classifier that never says "spam" has `tp + fp == 0`, so its precision can't be calculated, and the function returns 0.0.

In [5]:
def accuracy(tp, fp, fn, tn):
    return (tp + tn) / (tp + fp + fn + tn)


def precision(tp, fp):
    if tp + fp == 0:
        return 0.0  # the classifier never said "spam"
    return tp / (tp + fp)


def recall(tp, fn):
    if tp + fn == 0:
        return 0.0  # there were no real spam messages
    return tp / (tp + fn)


def f1(precision_value, recall_value):
    if precision_value + recall_value == 0:
        return 0.0
    return 2 * precision_value * recall_value / (precision_value + recall_value)


p = precision(tp, fp)
r = recall(tp, fn)
print("Accuracy: ", round(accuracy(tp, fp, fn, tn), 3))
print("Precision:", round(p, 3))
print("Recall:   ", round(r, 3))
print("F1:       ", round(f1(p, r), 3))

Accuracy:  0.986
Precision: 0.967
Recall:    0.932
F1:        0.949


## 6. Checking the Results Against scikit-learn

scikit-learn calculates the same performance metrics: `accuracy_score`, `precision_score`, `recall_score` and `f1_score` in `sklearn.metrics`. The function below prints our by-hand results next to scikit-learn's, so that you can see that they agree. We use it for the spam classifier and for the "always ham" classifier.

In [6]:
from sklearn.metrics import f1_score, precision_score, recall_score


def show_scores(name, true_labels, predicted_labels):
    tp, fp, fn, tn = count_results(true_labels, predicted_labels)
    p = precision(tp, fp)
    r = recall(tp, fn)
    print(name)
    print("  counts:       TP =", tp, " FP =", fp, " FN =", fn, " TN =", tn)
    print("  by hand:      accuracy", round(accuracy(tp, fp, fn, tn), 3),
          "| precision", round(p, 3), "| recall", round(r, 3), "| F1", round(f1(p, r), 3))
    print("  scikit-learn: accuracy", round(accuracy_score(true_labels, predicted_labels), 3),
          "| precision", round(precision_score(true_labels, predicted_labels, zero_division=0), 3),
          "| recall", round(recall_score(true_labels, predicted_labels), 3),
          "| F1", round(f1_score(true_labels, predicted_labels), 3))


show_scores("Spam classifier", test_labels, predicted_labels)
show_scores("Always ham", test_labels, always_ham)

Spam classifier
  counts:       TP = 178  FP = 6  FN = 13  TN = 1197
  by hand:      accuracy 0.986 | precision 0.967 | recall 0.932 | F1 0.949
  scikit-learn: accuracy 0.986 | precision 0.967 | recall 0.932 | F1 0.949
Always ham
  counts:       TP = 0  FP = 0  FN = 191  TN = 1203
  by hand:      accuracy 0.863 | precision 0.0 | recall 0.0 | F1 0.0
  scikit-learn: accuracy 0.863 | precision 0.0 | recall 0.0 | F1 0.0


## 7. Reading the Numbers

**The "always ham" classifier.** Its accuracy is 86.3%, but its recall is 0.0: it found none of the 191 spam messages. Its precision is also reported as 0.0, but really it is undefined, because the classifier never said "spam". When one class is rare, accuracy mostly measures how common the *other* class is, so it can't tell a useful classifier from a useless one. Precision and recall look at the rare class itself.

**The spam classifier.** Its accuracy of 98.6% is far above the 86.3% of "always ham", so the classifier really is useful. Precision and recall show what kind of mistakes it makes. It said "spam" for 184 of the 1,394 test messages (178 + 6). Its precision is 0.967 because 178 of those 184 really were spam, and the other 6 were normal messages that it wrongly called spam. Its recall is 0.932 because there were 191 spam messages in the test set, and it found 178 of them. The other 13 got through.

**Which mistake is worse?** That depends on what the classifier is for. For a spam filter, losing a real message (a false positive) is probably worse than letting some spam through (a false negative), so precision matters more. For a classifier that flags emergency messages, it would be the other way around, and recall would matter more. You choose the performance metrics to watch by thinking about what each kind of mistake costs.

## 8. The Classification Report and the Confusion Matrix

The Spam Detection example printed a classification report and a confusion matrix. Now you can read them. The report has the precision, recall and F1 of *each* class, treating that class as the positive one. The row for `spam` has the numbers that we calculated by hand. In the confusion matrix, each row is a true class and each column is a predicted class, so the numbers on the diagonal are the correct answers.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(test_labels, predicted_labels, target_names=["ham", "spam"]))
print("Confusion matrix (rows = true, columns = predicted):")
print(["ham", "spam"])
print(confusion_matrix(test_labels, predicted_labels))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99      1203
        spam       0.97      0.93      0.95       191

    accuracy                           0.99      1394
   macro avg       0.98      0.96      0.97      1394
weighted avg       0.99      0.99      0.99      1394

Confusion matrix (rows = true, columns = predicted):
['ham', 'spam']
[[1197    6]
 [  13  178]]


The row for really ham is `[1197 6]` and the row for really spam is `[13 178]`. Those are the same four counts as before.

## 9. More Than Two Classes

The spam classifier chooses between two classes. In the labs, some of the classifiers choose between several. scikit-learn then calculates precision, recall and F1 for each class in turn. To show what that looks like, here is a made-up example: a help desk sorts 100 messages into three departments. Security is the rare one: it gets only 10 of the 100 messages. Each case below is (true department, predicted department, how many messages).

In [8]:
cases = [
    ("Billing", "Billing", 57), ("Billing", "Technical", 3),
    ("Technical", "Technical", 27), ("Technical", "Billing", 1), ("Technical", "Security", 2),
    ("Security", "Security", 5), ("Security", "Billing", 3), ("Security", "Technical", 2),
]
true_departments = []
predicted_departments = []
for true_department, predicted_department, count in cases:
    for i in range(count):
        true_departments.append(true_department)
        predicted_departments.append(predicted_department)

names = ["Billing", "Technical", "Security"]
print(classification_report(true_departments, predicted_departments, labels=names))
print("Confusion matrix (rows = true, columns = predicted):")
print(names)
print(confusion_matrix(true_departments, predicted_departments, labels=names))

              precision    recall  f1-score   support

     Billing       0.93      0.95      0.94        60
   Technical       0.84      0.90      0.87        30
    Security       0.71      0.50      0.59        10

    accuracy                           0.89       100
   macro avg       0.83      0.78      0.80       100
weighted avg       0.89      0.89      0.89       100

Confusion matrix (rows = true, columns = predicted):
['Billing', 'Technical', 'Security']
[[57  3  0]
 [ 1 27  2]
 [ 3  2  5]]


At the bottom of the report are two kinds of average, and they tell different stories:

- **Macro average:** the plain average of the per-class scores. Every class counts the same, however small. Security's recall of 0.50 pulls the macro F1 down to 0.80.
- **Weighted average:** the per-class scores averaged in proportion to the number of messages in each class. The big departments dominate it, so the weighted F1 is 0.89, the same as the accuracy.

If you only looked at the accuracy, you wouldn't notice that half of the Security messages were sent to the wrong department. The confusion matrix shows where they went: 3 to Billing and 2 to Technical.

## Try It Yourself

- Make a classifier that calls every message spam: `always_spam = [1] * len(test_labels)`. Use `show_scores` on it. What are its precision and recall? Why is it useless even though its recall is perfect?
- In the first code cell, change `test_size` from 0.25 to 0.5 and run the notebook again. Which of the numbers change, and by how much? What does that tell you about how much to trust the result from one test set?
- Which mistake would you rather your spam filter make: sending a real message to the spam folder, or letting a spam message through? What do the precision and recall of spam tell you about that?
- In the three-department example, move one of the Security messages that was sent to Billing so it is predicted correctly. How much does the accuracy change? How much does the Security recall change?

*This notebook was drafted by Claude Code and is waiting for review by Brian Bird. 10/21/2026*